In [ ]:
# Parameters
num_qubits = 4
input_state = "0101"


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('agg')
import warnings
warnings.filterwarnings('ignore')


In [ ]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import circuit_drawer, plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector, partial_trace
import matplotlib.pyplot as plt
import numpy as np

num_qubits = int(num_qubits)
input_state = str(input_state)

if len(input_state) != num_qubits:
    input_state = '0' * num_qubits

print(f"QFT on {num_qubits} qubits, input |{input_state}⟩")

qc = QuantumCircuit(num_qubits)

# Set input state
for i, bit in enumerate(reversed(input_state)):
    if bit == '1':
        qc.x(i)
qc.barrier()

# QFT
for j in range(num_qubits):
    qc.h(j)
    for k in range(j+1, num_qubits):
        qc.cp(np.pi / (2**(k-j)), k, j)
# Swap
for i in range(num_qubits // 2):
    qc.swap(i, num_qubits - i - 1)

qc.measure_all()

fig = circuit_drawer(qc, output='mpl')
display(fig)
plt.close(fig)

simulator = Aer.get_backend('aer_simulator')
compiled = transpile(qc, simulator)
job = simulator.run(compiled, shots=1000)
result = job.result()
counts = result.get_counts()
print(f"Results: {counts}")

fig2 = plot_histogram(counts)
display(fig2)
plt.close(fig2)

# Bloch sphere for qubit 0
qc_sv = QuantumCircuit(num_qubits)
for i, bit in enumerate(reversed(input_state)):
    if bit == '1':
        qc_sv.x(i)
for j in range(num_qubits):
    qc_sv.h(j)
    for k in range(j+1, num_qubits):
        qc_sv.cp(np.pi / (2**(k-j)), k, j)
for i in range(num_qubits // 2):
    qc_sv.swap(i, num_qubits - i - 1)
sv = Statevector.from_instruction(qc_sv)
rho = partial_trace(sv, list(range(1, num_qubits)))
a = np.sqrt(np.real(rho.data[0, 0]))
b_complex = rho.data[1, 0] / a if a > 1e-6 else 0
theta = 2 * np.arccos(np.clip(a, 0, 1))
phi = np.angle(b_complex) % (2 * np.pi)

try:
    fig3 = plot_bloch_multivector(sv)
    display(fig3)
    plt.close(fig3)
except Exception:
    pass
print(f"BLOCH_THETA={theta:.6f}")
print(f"BLOCH_PHI={phi:.6f}")
